# Phase 3: Supervised Learning for False Positive Reduction
# Task 3.3: Tuned Linear SVM

Build and tune a supervised flow-level classifier that separates benign and attack flows.

This notebook uses the same fixed Phase 3 flow splits as the Random Forest and XGBoost notebooks. A linear SVM is used because kernel SVM does not scale well to ~200k flow rows; `CalibratedClassifierCV` wraps `LinearSVC` so ROC/PR metrics can use probability scores.


## Setup and load the fixed flow splits


In [ ]:

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    average_precision_score,
    precision_recall_curve,
    make_scorer
)

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.data.flow_preprocessing import ATTACK_COL, BINARY_COL, FLOW_KEY_COL, TARGET_COL, build_flow_preprocessor

DATA_FOLDER = "phase3"
DATA_DIR = PROJECT_ROOT / "data" / "processed" / DATA_FOLDER
META_PATH = DATA_DIR / "flow_split_metadata.json"

with open(META_PATH, "r") as f:
    metadata = json.load(f)

train_df = pd.read_csv(DATA_DIR / "flow_train.csv", low_memory=False)
validation_df = pd.read_csv(DATA_DIR / "flow_validation.csv", low_memory=False)
test_df = pd.read_csv(DATA_DIR / "flow_test.csv", low_memory=False)

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)
metadata


### Verify the class distribution and split integrity


In [ ]:
for name, current_df in [("Train", train_df), ("Validation", validation_df), ("Test", test_df)]:
    print(f"{name} binary counts")
    print(current_df[BINARY_COL].value_counts())
    print()

train_keys = set(train_df[FLOW_KEY_COL])
validation_keys = set(validation_df[FLOW_KEY_COL])
test_keys = set(test_df[FLOW_KEY_COL])

print("Flow-key overlap")
print("  train ∩ validation:", len(train_keys & validation_keys))
print("  train ∩ test:", len(train_keys & test_keys))
print("  validation ∩ test:", len(validation_keys & test_keys))


## SVM feature preparation


The model uses the numeric flow-behaviour features recorded in the split metadata. Labels, flow identifiers, IP addresses, timestamps, and source-file information are excluded.

Unlike tree models, SVM distance-based decisions need comparable feature scales, so the shared preprocessor enables `RobustScaler`.


In [ ]:
feature_cols = metadata["feature_columns"]
missing_features = [col for col in feature_cols if col not in train_df.columns]

if missing_features:
    raise ValueError(f"Missing model features: {missing_features}")

X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].to_numpy()

X_validation = validation_df[feature_cols]
y_validation = validation_df[TARGET_COL].to_numpy()

X_test = test_df[feature_cols]
y_test = test_df[TARGET_COL].to_numpy()

print("Input features:", len(feature_cols))
print("Training missing values:", int(X_train.isna().sum().sum()))
print("Validation missing values:", int(X_validation.isna().sum().sum()))
print("Test missing values:", int(X_test.isna().sum().sum()))


## Tune Linear SVM with 5-fold stratified cross-validation


The hyperparameter search uses only the training split.\n\n`StratifiedKFold` preserves class balance across folds. `LinearSVC` is configured with `dual=False` (appropriate when samples >> features) and wrapped in `CalibratedClassifierCV` so `predict_proba` is available for thresholded evaluation and curve metrics.\n

In [ ]:
RANDOM_STATE = 1
N_ITER = 8
CLASSIFICATION_THRESHOLD = 0.50

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

preprocessor = build_flow_preprocessor(scale=True)
# dual=False is preferred when n_samples >> n_features and converges far more reliably
# on this ~90k-row flow dataset than dual="auto"/liblinear hinge solvers.
base_svm = LinearSVC(
    class_weight="balanced",
    dual=False,
    loss="squared_hinge",
    penalty="l2",
    max_iter=20000,
    random_state=RANDOM_STATE
)
calibrated_svm = CalibratedClassifierCV(estimator=base_svm, method="sigmoid", cv=2)

svm_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", calibrated_svm)
])

param_distributions = {
    "model__estimator__C": [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0],
    "model__method": ["sigmoid", "isotonic"]
}

scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "f1": make_scorer(f1_score, zero_division=0),
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0)
}

svm_search = RandomizedSearchCV(
    estimator=svm_pipeline,
    param_distributions=param_distributions,
    n_iter=N_ITER,
    scoring=scoring,
    refit="pr_auc",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    random_state=RANDOM_STATE,
    return_train_score=False
)

start_search = time.perf_counter()
svm_search.fit(X_train, y_train)
search_time = time.perf_counter() - start_search

svm_pipeline = svm_search.best_estimator_

print(f"Search time: {search_time:.2f} seconds")
print(f"Best mean CV PR-AUC: {svm_search.best_score_:.4f}")
print("Best parameters:")
for name, value in svm_search.best_params_.items():
    print(f"  {name}: {value}")

processed_feature_count = svm_pipeline.named_steps["preprocessor"].named_steps["remove_constant"].get_support().sum()
print("Processed features:", processed_feature_count)


### Cross-validation results

The table below shows the highest-ranked sampled configurations. The final estimator is the row with rank 1 for mean validation PR-AUC across the five folds.


In [ ]:
cv_results_df = pd.DataFrame(svm_search.cv_results_)

result_columns = [
    "rank_test_pr_auc",
    "mean_test_pr_auc",
    "std_test_pr_auc",
    "mean_test_roc_auc",
    "mean_test_f1",
    "mean_test_precision",
    "mean_test_recall",
    "mean_fit_time",
    "param_model__estimator__C",
    "param_model__method"
]

top_cv_results = cv_results_df[result_columns].sort_values("rank_test_pr_auc").head(10)
display(top_cv_results.round(4))


## Evaluate the tuned SVM


In [ ]:
def evaluate_binary_classifier(name, y_true, scores, threshold, prediction_time):
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    result = {
        "dataset": name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp),
        "fpr": fp / (fp + tn),
        "fnr": fn / (fn + tp),
        "roc_auc": roc_auc_score(y_true, scores),
        "pr_auc": average_precision_score(y_true, scores),
        "alerts": int(y_pred.sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "prediction_time_sec": prediction_time
    }

    return result, y_pred

start_validation = time.perf_counter()
validation_scores = svm_pipeline.predict_proba(X_validation)[:, 1]
validation_prediction_time = time.perf_counter() - start_validation

start_test = time.perf_counter()
test_scores = svm_pipeline.predict_proba(X_test)[:, 1]
test_prediction_time = time.perf_counter() - start_test

validation_metrics, validation_pred = evaluate_binary_classifier(
    "Validation", y_validation, validation_scores, CLASSIFICATION_THRESHOLD, validation_prediction_time
)
test_metrics, test_pred = evaluate_binary_classifier(
    "Test", y_test, test_scores, CLASSIFICATION_THRESHOLD, test_prediction_time
)

metrics_df = pd.DataFrame([validation_metrics, test_metrics])
display(metrics_df.round(4))


### Confusion matrices

The selected hyperparameters are evaluated on the untouched validation set and then on the test set. The probability threshold remains at 0.5 so that hyperparameter tuning and threshold choice stay separate.


In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    plt.figure(figsize=(5, 4))
    plt.imshow(cm)
    plt.title(title)
    plt.colorbar()
    plt.xticks([0, 1], ["Benign", "Attack"])
    plt.yticks([0, 1], ["Benign", "Attack"])

    for i in range(2):
        for j in range(2):
            plt.text(j, i, f"{cm[i, j]:,}", ha="center", va="center")

    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(y_validation, validation_pred, "Tuned Linear SVM Validation Confusion Matrix")
plot_confusion_matrix(y_test, test_pred, "Tuned Linear SVM Test Confusion Matrix")


### Test classification report


In [ ]:
report = classification_report(
    y_test,
    test_pred,
    target_names=["Benign", "Attack"],
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).T
display(report_df.round(4))


### Results by traffic type


In [ ]:
type_results = test_df[[ATTACK_COL]].copy()
type_results["y_true"] = y_test
type_results["y_pred"] = test_pred
type_results["correct"] = type_results["y_true"] == type_results["y_pred"]

result_by_type = type_results.groupby(ATTACK_COL).agg(total=("correct", "size"), correct=("correct", "sum"))
result_by_type["incorrect"] = result_by_type["total"] - result_by_type["correct"]
result_by_type["correct_rate"] = result_by_type["correct"] / result_by_type["total"] * 100
result_by_type["incorrect_rate"] = 100 - result_by_type["correct_rate"]

type_order = ["Benign", "DDoS-HTTP Flood", "DoS-HTTP Flood", "DNS Spoofing", "Brute Force", "XSS"]
result_by_type = result_by_type.reindex(type_order).dropna()

display(result_by_type[["total", "correct", "incorrect", "correct_rate"]].round(2))

plot_data = result_by_type[["correct_rate", "incorrect_rate"]]
ax = plot_data.plot(kind="barh", stacked=True, figsize=(11, 6))

for i, (_, row) in enumerate(result_by_type.iterrows()):
    count_text = f'Correct: {int(row["correct"]):,}   Incorrect: {int(row["incorrect"]):,}'
    ax.text(102, i, count_text, va="center", fontsize=9)

ax.set_xlim(0, 145)
ax.set_xlabel("Percentage of flows")
ax.set_ylabel("Traffic type")
ax.set_title("Correct and Incorrect Tuned Linear SVM Classification by Traffic Type")
ax.legend(["Correct", "Incorrect"], loc="lower right")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


### ROC and precision-recall curves


In [ ]:
def plot_roc_curve(y_true, scores, title):
    fpr, tpr, _ = roc_curve(y_true, scores)
    auc_value = roc_auc_score(y_true, scores)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"ROC-AUC = {auc_value:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()


def plot_precision_recall_curve(y_true, scores, title):
    precision, recall, _ = precision_recall_curve(y_true, scores)
    pr_auc = average_precision_score(y_true, scores)

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, label=f"PR-AUC = {pr_auc:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

plot_roc_curve(y_test, test_scores, "Tuned Linear SVM Test ROC Curve")
plot_precision_recall_curve(y_test, test_scores, "Tuned Linear SVM Test Precision-Recall Curve")


## Linear SVM coefficient magnitudes


Linear SVM does not expose tree-style feature importances. Absolute mean coefficients from the calibrated `LinearSVC` estimators are used instead as a simple ranking of which scaled features push the decision boundary.


In [ ]:
processed_feature_names = svm_pipeline.named_steps["preprocessor"].get_feature_names_out(feature_cols)
calibrated = svm_pipeline.named_steps["model"]
coef_stack = np.vstack([clf.estimator.coef_.ravel() for clf in calibrated.calibrated_classifiers_])
mean_abs_coef = np.abs(coef_stack).mean(axis=0)

importance_df = pd.DataFrame({
    "feature": processed_feature_names,
    "mean_abs_coefficient": mean_abs_coef
}).sort_values("mean_abs_coefficient", ascending=False).reset_index(drop=True)

display(importance_df.head(20))

plot_importance = importance_df.head(20).sort_values("mean_abs_coefficient")
plt.figure(figsize=(9, 7))
plt.barh(plot_importance["feature"], plot_importance["mean_abs_coefficient"])
plt.xlabel("Mean absolute coefficient")
plt.ylabel("Feature")
plt.title("Top 20 Tuned Linear SVM Features")
plt.tight_layout()
plt.show()


## Save the tuned results


In [ ]:
RESULTS_DIR = PROJECT_ROOT / "reports" / "phase3"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

metrics_output = metrics_df.copy()
metrics_output.insert(0, "model", "Linear SVM Tuned")
metrics_output["search_time_sec"] = search_time
metrics_output["refit_time_sec"] = svm_search.refit_time_
metrics_output.to_csv(RESULTS_DIR / "svm_tuned_metrics.csv", index=False)

importance_df.to_csv(RESULTS_DIR / "svm_tuned_feature_importance.csv", index=False)
cv_results_df.to_csv(RESULTS_DIR / "svm_random_search_results.csv", index=False)

best_parameters = {
    "best_cv_pr_auc": float(svm_search.best_score_),
    "search_time_sec": search_time,
    "refit_time_sec": svm_search.refit_time_,
    "n_iter": N_ITER,
    "cv_folds": cv.get_n_splits(),
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "best_params": svm_search.best_params_
}

with open(RESULTS_DIR / "svm_best_parameters.json", "w") as f:
    json.dump(best_parameters, f, indent=2)

prediction_cols = [FLOW_KEY_COL, ATTACK_COL, BINARY_COL, TARGET_COL]
test_predictions = test_df[prediction_cols].copy()
test_predictions["attack_probability"] = test_scores
test_predictions["prediction"] = test_pred
test_predictions.to_csv(RESULTS_DIR / "svm_tuned_test_predictions.csv", index=False)

print("Saved tuned Linear SVM results to:", RESULTS_DIR)


## Final tuned Linear SVM summary


In [ ]:
test_row = metrics_df.loc[metrics_df["dataset"] == "Test"].iloc[0]

print("Final tuned Linear SVM test results")
print("-----------------------------------")
print(f"Best mean CV PR-AUC: {svm_search.best_score_:.4f}")
print(f"Accuracy: {test_row['accuracy']:.4f}")
print(f"Precision: {test_row['precision']:.4f}")
print(f"Recall: {test_row['recall']:.4f}")
print(f"F1-score: {test_row['f1']:.4f}")
print(f"ROC-AUC: {test_row['roc_auc']:.4f}")
print(f"PR-AUC: {test_row['pr_auc']:.4f}")
print(f"FPR: {test_row['fpr']:.4f}")
print(f"FNR: {test_row['fnr']:.4f}")
print(f"Search time: {search_time:.2f} seconds")
print(f"Best-model refit time: {svm_search.refit_time_:.2f} seconds")
print(f"Test prediction time: {test_prediction_time:.2f} seconds")

print()
print("Best hyperparameters")
for name, value in svm_search.best_params_.items():
    print(f"{name}: {value}")

print()
print("Correct classification rate by traffic type")
for attack_type in type_order:
    row = result_by_type.loc[attack_type]
    print(f"{attack_type}: {row['correct_rate']:.2f}%")
